In [ ]:

# ============================================================
# CELL 1: INITIALIZE SNOWPARK SESSION
# ============================================================

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col

session = get_active_session()

# Set database and schema context
session.sql("""
    USE DATABASE API_DATA_ENGINEERING
""").collect()

print("Snowpark session initialized successfully")
print("Database: API_DATA_ENGINEERING")
print("Runtime: Snowflake Warehouse Runtime")


In [ ]:

# ============================================================
# CELL 2: PIPELINE CONFIGURATION
# ============================================================

DATABASE = "API_DATA_ENGINEERING"

SOURCE_TABLE = "AI_RAW.CUSTOMER_FEEDBACK"

TARGET_TABLE = "AI_ENRICHED.CUSTOMER_FEEDBACK_AI"

ANALYTICS_VIEW = "AI_ANALYTICS.V_CUSTOMER_FEEDBACK_SUMMARY"

# Process only 20 records initially.
# After successful testing, increase to 120.
BATCH_LIMIT = 20

# AI classification categories
FEEDBACK_CATEGORIES = [
    "Product Quality",
    "Delivery",
    "Customer Support",
    "Pricing",
    "Product Performance",
    "Returns and Refunds",
    "Other"
]

print("Pipeline configuration loaded")
print("Batch size:", BATCH_LIMIT)
print("AI categories:", FEEDBACK_CATEGORIES)


In [ ]:

# ============================================================
# CELL 3: READ RAW CUSTOMER FEEDBACK
# ============================================================

raw_df = session.table(SOURCE_TABLE)

# Read previously enriched IDs to prevent duplicates
existing_df = (
    session.table(TARGET_TABLE)
    .select("FEEDBACK_ID")
    .distinct()
)

# Find records that have not been processed
pending_df = (
    raw_df
    .filter(col("IS_PROCESSED") == False)
    .join(
        existing_df,
        raw_df["FEEDBACK_ID"] == existing_df["FEEDBACK_ID"],
        "left_anti"
    )
    .select(
        raw_df["FEEDBACK_ID"],
        raw_df["CUSTOMER_ID"],
        raw_df["CUSTOMER_NAME"],
        raw_df["PRODUCT_NAME"],
        raw_df["FEEDBACK_TEXT"],
        raw_df["SOURCE_CHANNEL"],
        raw_df["CREATED_AT"]
    )
    .limit(BATCH_LIMIT)
)

print("Pending records in this batch:")
pending_df.count()


In [ ]:

# ============================================================
# CELL 4: LOAD FEEDBACK + ADVANCED SENTIMENT ANALYSIS
# ============================================================

import re
import json
from datetime import datetime, timezone
from snowflake.snowpark.context import get_active_session

session = get_active_session()

SOURCE_TABLE = "API_DATA_ENGINEERING.AI_RAW.CUSTOMER_FEEDBACK"
TARGET_TABLE = "API_DATA_ENGINEERING.AI_ENRICHED.CUSTOMER_FEEDBACK_AI"

feedback_df = (
    session.table(SOURCE_TABLE)
    .filter("IS_PROCESSED = FALSE")
)

feedback_rows = feedback_df.collect()

print("Feedback records loaded:", len(feedback_rows))

# Weighted sentiment vocabulary
positive_words = {
    "good": 1, "great": 2, "excellent": 3,
    "happy": 2, "love": 3, "perfect": 3,
    "amazing": 3, "satisfied": 2, "wonderful": 3,
    "recommend": 2, "fast": 1, "helpful": 2,
    "quality": 1, "fantastic": 3, "awesome": 3,
    "easy": 1, "friendly": 2, "smooth": 1,
    "reliable": 2, "comfortable": 2
}

negative_words = {
    "bad": -2, "poor": -2, "terrible": -3,
    "disappointed": -2, "slow": -1, "broken": -3,
    "damaged": -2, "defective": -3, "expensive": -1,
    "hate": -3, "worst": -3, "late": -2,
    "problem": -1, "issue": -1, "unhappy": -2,
    "difficult": -1, "unfriendly": -2,
    "failure": -2, "faulty": -3, "unacceptable": -3
}

negation_words = {
    "not", "no", "never", "hardly", "isn't",
    "wasn't", "doesn't", "didn't", "cannot"
}

intensifiers = {
    "very": 1.5,
    "extremely": 2.0,
    "really": 1.5,
    "highly": 1.5,
    "slightly": 0.5
}

def analyse_sentiment(text):

    words = re.findall(r"\b[\w']+\b", (text or "").lower())

    total_score = 0
    positive_score = 0
    negative_score = 0

    for i, word in enumerate(words):

        value = positive_words.get(
            word, negative_words.get(word, 0)
        )

        if value == 0:
            continue

        # Look for negation in the preceding three words
        previous_words = words[max(0, i-3):i]

        if any(w in negation_words for w in previous_words):
            value *= -1

        # Apply intensity adjustment
        if i > 0 and words[i-1] in intensifiers:
            value *= intensifiers[words[i-1]]

        total_score += value

        if value > 0:
            positive_score += value
        else:
            negative_score += abs(value)

    denominator = positive_score + negative_score

    if denominator == 0:
        return "NEUTRAL", 0.0, "NEUTRAL"

    score = (positive_score - negative_score) / denominator

    if positive_score > 0 and negative_score > 0:
        sentiment = "MIXED"
    elif score >= 0.25:
        sentiment = "POSITIVE"
    elif score <= -0.25:
        sentiment = "NEGATIVE"
    else:
        sentiment = "NEUTRAL"

    return sentiment, round(score, 3), sentiment


# Execute sentiment analysis
sentiment_results = []

for row in feedback_rows:

    sentiment, score, sentiment_type = analyse_sentiment(
        row["FEEDBACK_TEXT"]
    )

    sentiment_results.append({
        "FEEDBACK_ID": row["FEEDBACK_ID"],
        "SENTIMENT": sentiment,
        "SENTIMENT_SCORE": score,
        "SENTIMENT_TYPE": sentiment_type
    })

print("Sentiment analysis completed")
print("Records analysed:", len(sentiment_results))

for result in sentiment_results[:10]:
    print(result)


In [ ]:

# ============================================================
# CELL 5: MULTI-LABEL FEEDBACK CLASSIFICATION
# ============================================================

category_keywords = {

    "PRODUCT_QUALITY": [
        "quality", "defective", "damaged", "broken",
        "durable", "material", "faulty", "reliable",
        "excellent product", "poor product"
    ],

    "DELIVERY": [
        "delivery", "shipping", "late", "delayed",
        "arrived", "package", "shipment", "fast delivery"
    ],

    "PRICE": [
        "price", "expensive", "cheap", "cost",
        "value", "discount", "affordable"
    ],

    "CUSTOMER_SERVICE": [
        "support", "service", "staff", "response",
        "help", "team", "communication"
    ],

    "PRODUCT_FEATURES": [
        "features", "design", "performance",
        "functionality", "size", "comfortable"
    ]
}

def classify_feedback(text):

    text = (text or "").lower()

    category_scores = {}

    for category, keywords in category_keywords.items():

        score = 0

        for keyword in keywords:

            if keyword in text:
                # Longer phrases receive greater weight
                score += 2 if " " in keyword else 1

        category_scores[category] = score

    maximum = max(category_scores.values(), default=0)

    if maximum == 0:
        return ["GENERAL_FEEDBACK"], 0.0

    # Include every category with a positive score
    detected_categories = [
        category
        for category, score in category_scores.items()
        if score > 0
    ]

    confidence = maximum / max(
        sum(category_scores.values()), 1
    )

    return detected_categories, round(confidence, 3)


classification_results = []

for row in feedback_rows:

    categories, confidence = classify_feedback(
        row["FEEDBACK_TEXT"]
    )

    classification_results.append({
        "FEEDBACK_ID": row["FEEDBACK_ID"],
        "FEEDBACK_CATEGORY": categories,
        "CLASSIFICATION_CONFIDENCE": confidence
    })

print("Multi-label classification completed")
print("Records classified:", len(classification_results))

for result in classification_results[:100]:
    print(result)


In [ ]:

# ============================================================
# CELL 6: STRUCTURED INFORMATION EXTRACTION
# ============================================================

issue_keywords = {
    "DELIVERY_ISSUE": [
        "delivery", "shipping", "late", "delayed"
    ],
    "QUALITY_ISSUE": [
        "defective", "broken", "damaged", "faulty"
    ],
    "PRICE_ISSUE": [
        "expensive", "price", "cost", "overpriced"
    ],
    "SERVICE_ISSUE": [
        "support", "service", "staff", "response"
    ],
    "PERFORMANCE_ISSUE": [
        "performance", "slow", "not working", "failure"
    ]
}

action_keywords = {
    "REFUND": ["refund", "money back"],
    "REPLACEMENT": ["replace", "replacement", "exchange"],
    "REPAIR": ["repair", "fix"],
    "SUPPORT": ["contact", "help", "assistance"],
    "COMPLAINT": ["complaint", "complain", "unacceptable"]
}

def extract_information(text):

    text_lower = (text or "").lower()

    detected_issues = []
    detected_actions = []

    for issue, keywords in issue_keywords.items():

        if any(keyword in text_lower for keyword in keywords):
            detected_issues.append(issue)

    for action, keywords in action_keywords.items():

        if any(keyword in text_lower for keyword in keywords):
            detected_actions.append(action)

    if not detected_issues:
        detected_issues = ["NO_SPECIFIC_ISSUE"]

    if not detected_actions:
        detected_actions = ["NONE"]

    return {
        "issues": detected_issues,
        "requested_actions": detected_actions,
        "product_name": None
    }


extraction_results = []

for row in feedback_rows:

    extracted = extract_information(row["FEEDBACK_TEXT"])

    # Use the actual product name from the source record
    extracted["product_name"] = row["PRODUCT_NAME"]

    extraction_results.append({
        "FEEDBACK_ID": row["FEEDBACK_ID"],
        "EXTRACTED_INFORMATION": extracted
    })

print("Information extraction completed")
print("Records processed:", len(extraction_results))

for result in extraction_results[:100]:
    print(result)


In [ ]:

# ============================================================
# CELL 7: EXTRACTIVE FEEDBACK SUMMARISATION
# ============================================================

stop_words = {
    "the", "and", "for", "with", "this", "that",
    "was", "were", "are", "is", "very", "but",
    "have", "has", "had", "from", "they", "them",
    "you", "your", "our", "not", "too"
}

def summarize_feedback(text):

    if not text:
        return ""

    sentences = re.split(r"(?<=[.!?])\s+", text.strip())

    if len(sentences) <= 1:
        return text[:500]

    sentence_scores = []

    for sentence in sentences:

        words = re.findall(
            r"\b[a-zA-Z]+\b", sentence.lower()
        )

        meaningful_words = [
            word for word in words
            if word not in stop_words
        ]

        score = len(set(meaningful_words))

        sentence_scores.append((score, sentence))

    best_sentence = max(
        sentence_scores,
        key=lambda item: item[0]
    )[1]

    return best_sentence[:500]


summary_results = []

for row in feedback_rows:

    summary = summarize_feedback(row["FEEDBACK_TEXT"])

    summary_results.append({
        "FEEDBACK_ID": row["FEEDBACK_ID"],
        "AI_SUMMARY": summary
    })

print("Summarisation completed")
print("Records summarised:", len(summary_results))

for result in summary_results[:10]:
    print(result)


In [ ]:

# ============================================================
# CELL 8: COMBINE ALL ENRICHMENT RESULTS
# ============================================================

from datetime import datetime
import json

# Create lookup dictionaries using FEEDBACK_ID
sentiment_lookup = {
    item["FEEDBACK_ID"]: item
    for item in sentiment_results
}

classification_lookup = {
    item["FEEDBACK_ID"]: item
    for item in classification_results
}

extraction_lookup = {
    item["FEEDBACK_ID"]: item
    for item in extraction_results
}

summary_lookup = {
    item["FEEDBACK_ID"]: item
    for item in summary_results
}

# Prepare final enriched records
final_records = []

for row in feedback_df.collect():

    feedback_id = row["FEEDBACK_ID"]

    sentiment_data = sentiment_lookup[feedback_id]
    classification_data = classification_lookup[feedback_id]
    extraction_data = extraction_lookup[feedback_id]
    summary_data = summary_lookup[feedback_id]

    final_records.append({

        "FEEDBACK_ID": feedback_id,
        "CUSTOMER_ID": row["CUSTOMER_ID"],
        "CUSTOMER_NAME": row["CUSTOMER_NAME"],
        "PRODUCT_NAME": row["PRODUCT_NAME"],
        "FEEDBACK_TEXT": row["FEEDBACK_TEXT"],

        "SENTIMENT": sentiment_data["SENTIMENT"],
        "SENTIMENT_SCORE": sentiment_data["SENTIMENT_SCORE"],

        "FEEDBACK_CATEGORY": classification_data["FEEDBACK_CATEGORY"],

        "EXTRACTED_INFORMATION": json.dumps(
            extraction_data["EXTRACTED_INFORMATION"]
        ),

        "AI_SUMMARY": summary_data["AI_SUMMARY"],

        "AI_PROCESSED_AT": datetime.utcnow(),

        "MODEL_NAME": "PYTHON_RULE_BASED_NLP"

    })

print("Enrichment results combined")
print("Final records prepared:", len(final_records))

for record in final_records[:100]:
    print(record)
